data: full + partial vacate order

process 
1. **extract the total numbers by CD from the raw data file (excel)**
create `FULL + PARTIAL` by CD, making a dict like `{201: 8, 202: 12, ...}`. 

2. **open geojson** — read Community Districts border file.

3. **merge** — geojson의 각 CD마다 `boro_cd` 값을 보고, 1번에서 만든 딕셔너리에서 해당 숫자를 찾아 properties에 `vacate_total` 같은 새 필드로 추가한다. 매칭 안 되는 CD(공원, 공항 등 데이터 없는 구역)는 0으로.

4. **새 geojson으로 저장** — 결과를 새 파일로 쓴다. 이게 튜토리얼에 넣을 최종 파일.

핵심은 **1번과 3번을 잇는 열쇠가 CB번호 ↔ `boro_cd`**라는 거야. 아까 계속 강조한 그 부분.


### step 1: extract the needed data from xlsx file!

In [6]:
import openpyxl

wb = openpyxl.load_workbook(
    'Foil_2026-810-02410_-_Vacate_Orders_and_Complaints_for_Illegal_Basement_2020_to_Present.xlsx',
    read_only=True
)
ws = wb['Sheet1']

# first digit of CB stands for each boro
BORO_NAMES = {
    '1': 'Manhattan',
    '2': 'Bronx',
    '3': 'Brooklyn',
    '4': 'Queens',
    '5': 'Staten Island'
}

totals = {}          # {CB(str): full+partial}

# the table starts at row 15 (row 14 is the header)
for r in ws.iter_rows(min_row=15, values_only=True):
    boro, cb, full, partial = r[0], r[1], r[2], r[3]

    cb = str(cb).strip()          # strip.() removes empty space

    # condition: CB value passes when it is made only of digits and its length is three (CB is three digit word). 
    if not (cb.isdigit() and len(cb) == 3):
        continue                  # skip the row when not meeting the condition (header, empty cell, BIN, etc.)

    f = full or 0
    p = partial or 0

    boro_name = BORO_NAMES[cb[0]]          # seek boro name using the first digit

    totals[cb] = {
        'total': f + p,     # full + partial
        'boro': boro_name
    }            


print('뽑힌 CD 개수:', len(totals))
for k, v in list(totals.items())[:5]:      # 앞 5개만 확인용 출력
    print(k, v)

뽑힌 CD 개수: 57
201 {'total': 8, 'boro': 'Bronx'}
202 {'total': 12, 'boro': 'Bronx'}
203 {'total': 15, 'boro': 'Bronx'}
204 {'total': 24, 'boro': 'Bronx'}
205 {'total': 21, 'boro': 'Bronx'}


### step 2: merge this data with Community District geojson!

In [7]:
import json

# 1) open geojson
with open('Community_Districts_20260704.geojson') as f:
    gj = json.load(f)

# 2) merge totals with eac CD polygon
for feat in gj['features']:
    cd = feat['properties']['boro_cd']      # the polygon's CB number (예: '410')

    if cd in totals:                        # totals에 있으면 그 값을 사용
        feat['properties']['vacate_total'] = totals[cd]['total']
        feat['properties']['boro_name']    = totals[cd]['boro']
    else:                                   # 없으면 0으로 채움 (14개 CD)
        feat['properties']['vacate_total'] = 0
        feat['properties']['boro_name']    = BORO_NAMES.get(cd[0], '')   #
# 3) save as a new file
with open('basement_map.geojson', 'w') as f:
    json.dump(gj, f)

print('완료. 붙은 CD 수:', sum(1 for x in gj['features'] if x['properties']['vacate_total'] > 0))
print('전체 CD 수:', len(gj['features']))

완료. 붙은 CD 수: 57
전체 CD 수: 71


In [8]:
import json
gj = json.load(open('basement_map.geojson'))
p = gj['features'][0]['properties']
print('첫 feature properties:')
for k, v in p.items():
    print(f'  {k!r}: {v!r}')

첫 feature properties:
  ':id': 'row-v5e2_ukwx.qq4r'
  ':version': 'rv-8vaq.c5bx_k4nx'
  ':created_at': '2026-05-26T19:36:28.434Z'
  ':updated_at': '2026-05-26T19:36:28.434Z'
  'boro_cd': '410'
  'shape_leng': '105822.37731'
  'shape_area': '172077374.82'
  'vacate_total': 209
  'boro_name': 'Queens'
